# 06 · Qwen3.5-27B QLoRA 파인튜닝 (A100 40GB)

**목표**: 앙상블에 들어갈 "강한 파인튜닝 대형 모델" 하나 확보. 대회 Discussion에서 단독 최고로 보고된 것이 Qwen3.5-27B 파인튜닝(Public 0.97169)이었고, 저희는 대형 모델을 제로샷으로만 써왔음.

**설계**
- 4bit(nf4) QLoRA, LoRA r16 · α32, 언어 모델의 어텐션·선형어텐션·MLP 투영층
- 레시피는 EXP-102(8B, valid 650/672)와 동일: 정답 토큰에만 loss, 선지 순서 섞기, thinking 끔
- **프롬프트 2회 반복**(학습·추론 모두, 참고: Prompt Repetition Improves Non-Reasoning LLMs, arXiv 2512.14982)
- **추론 시 선지 순서 2가지(0칸·2칸 이동) 평균** (대회 Discussion에서 보고된 위치 편향 완화)
- dev는 제외(응답이 갈린 문항이 많아 train·test와 성격이 다름), 정제 train에서 valid 672 제외
- **학습 데이터 양은 속도를 잰 뒤 자동 결정**: 정답(a~d) 비율을 유지하는 층화 순서로 앞에서부터 N개 사용
- 50 step마다 체크포인트, 끊기면 모델 셀부터 다시 실행하면 이어서 학습

**순서**: 설치 → 데이터 → 모델 → 프롬프트 → ①점검(생성 3문항) → ②속도 측정·N 결정 → 학습 → valid → test

In [ ]:
# ===== 0. 설치 (실행 후 '런타임 → 세션 다시 시작') =====
!pip install -q -U transformers accelerate peft bitsandbytes
!pip install -q flash-linear-attention        # 선형 어텐션(Gated DeltaNet) 빠른 커널
!pip uninstall -y -q torchao
!apt-get install -y -qq p7zip-full > /dev/null

In [ ]:
# ===== 1. 데이터 =====
import os, glob, shutil, subprocess, time, random, math
import numpy as np, pandas as pd
from google.colab import drive
drive.mount("/content/drive")
SHARE_DIR = "/content/drive/MyDrive/ssafy/ai-challenge"
EXP_DIR = f"{SHARE_DIR}/experiment_data"
DATA_ROOT = "/content/data"
cnt = lambda d: len(glob.glob(f"{DATA_ROOT}/{d}/*.jpg"))
if cnt("train") < 6714 or cnt("test") < 6714:
    shutil.copy(glob.glob(f"{SHARE_DIR}/ssafy-16-2-ai*.zip")[0], "/content/data.zip")
    shutil.rmtree(DATA_ROOT, ignore_errors=True); os.makedirs(DATA_ROOT)
    subprocess.run(["7z","x","-y","-pocrisnotenough",f"-o{DATA_ROOT}","/content/data.zip"], check=True, stdout=subprocess.DEVNULL)
print("이미지 수 → train:", cnt("train"), "test:", cnt("test"))
resolve = lambda p: p if os.path.isabs(str(p)) else os.path.join(DATA_ROOT, str(p))
COLS = ["id","path","question","a","b","c","d","answer"]

valid_df = pd.read_csv(f"{SHARE_DIR}/split/valid_split_seed43.csv")[COLS]
clean = pd.concat([pd.read_csv(f"{SHARE_DIR}/split/train_clean_v2_seed43.csv"), pd.read_csv(f"{SHARE_DIR}/split/valid_clean_v2_seed43.csv")])[COLS]
train_df = clean[~clean.id.isin(set(valid_df.id))].reset_index(drop=True)
test_df = pd.read_csv(f"{DATA_ROOT}/test.csv")
for d in (valid_df, train_df, test_df): d["path"] = d["path"].map(resolve)
assert not set(train_df.id) & set(valid_df.id)

# 층화 순서: 정답별로 섞은 뒤 a,b,c,d를 번갈아 배치 → 앞에서 N개를 잘라도 정답 비율 유지
SEED = 43; rng = np.random.default_rng(SEED)
groups = {a: list(rng.permutation(train_df.index[train_df.answer == a])) for a in "abcd"}
order_all = []
while any(groups.values()):
    for a in "abcd":
        if groups[a]: order_all.append(groups[a].pop())
print("train 후보", len(train_df), "| valid", len(valid_df), "| test", len(test_df))

In [ ]:
# ===== 2. 모델 (다운로드 약 54GB, 5~10분) =====
import torch, torch.nn.functional as F
from PIL import Image
from transformers import AutoProcessor, BitsAndBytesConfig, get_cosine_schedule_with_warmup
try:
    from transformers import AutoModelForMultimodalLM as AutoVLM
except ImportError:
    from transformers import AutoModelForImageTextToText as AutoVLM
from peft import LoraConfig, get_peft_model, PeftModel, prepare_model_for_kbit_training
Image.MAX_IMAGE_PIXELS = None

MODEL_ID = "Qwen/Qwen3.5-27B"
TAG = "EXP-123_qwen3.5-27b_qlora"
OUT_DIR = f"{SHARE_DIR}/repro"; os.makedirs(OUT_DIR, exist_ok=True)
CKPT_DIR = f"{OUT_DIR}/models/{TAG}"
TRAIN_MAX_PIXELS = 768*32*32      # 학습 이미지 최대 약 768 토큰 (속도)
INFER_MAX_PIXELS = 1024*32*32     # 추론은 조금 더 크게
random.seed(SEED); torch.manual_seed(SEED)

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoVLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="cuda", dtype=torch.bfloat16, attn_implementation="sdpa")
processor = AutoProcessor.from_pretrained(MODEL_ID)
tok = processor.tokenizer
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False})
model.config.use_cache = False

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj|out_proj|in_proj_qkvz|gate_proj|up_proj|down_proj)")
START_STEP = 0
latest = sorted([p for p in glob.glob(f"{CKPT_DIR}/step_*") if os.path.exists(f"{p}/adapter_model.safetensors")], key=lambda p: int(p.split("_")[-1]))
if latest:
    model = PeftModel.from_pretrained(model, latest[-1], is_trainable=True); START_STEP = int(latest[-1].split("_")[-1]); print("재개:", latest[-1])
else:
    model = get_peft_model(model, lora)
model.print_trainable_parameters()
n_lora = sum(1 for n, m in model.named_modules() if hasattr(m, "lora_A"))
print("LoRA 모듈 수:", n_lora, "| GPU 사용", round(torch.cuda.memory_allocated()/1e9, 1), "GB"); assert n_lora > 0

In [ ]:
# ===== 3. 프롬프트 · 선지 섞기 · 정답 토큰 loss (EXP-102와 동일, thinking 끔) =====
def fit(img, maxp):
    w, h = img.size
    if w*h > maxp:
        s = (maxp/(w*h))**0.5; img = img.resize((max(32,int(w*s)), max(32,int(h*s))), Image.BICUBIC)
    return img

PROMPT_REPEAT = 2      # 같은 질문·선지를 2번 반복 (Prompt Repetition 논문, arXiv 2512.14982)
SHIFTS = [0, 2]        # 추론 시 선지 순서를 0칸·2칸 순환 이동해 두 번 풀고 원래 순서로 되돌려 평균

def prompt(q, opts):
    one = (f"이미지를 자세히 보고 질문에 답하세요. 이미지 속 글자를 정확히 읽는 것이 중요합니다.\n\n"
           f"질문: {q}\n" + "".join(f"{l}. {o}\n" for l, o in zip("abcd", opts)) +
           "\n정답 선지의 알파벳(a, b, c, d) 한 글자만 출력하세요.")
    return "\n\n".join([one] * PROMPT_REPEAT)

def chat(img, text):
    msgs = [{"role":"user","content":[{"type":"image","image":img},{"type":"text","text":text}]}]
    return processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

def build_train_example(r, shuffle=True):
    opts = [str(r[c]) for c in "abcd"]; gold = "abcd".index(str(r["answer"]).strip().lower())
    order = list(range(4))
    if shuffle: random.shuffle(order)
    new_opts = [opts[i] for i in order]; new_gold = "abcd"[order.index(gold)]
    img = fit(Image.open(r["path"]).convert("RGB"), TRAIN_MAX_PIXELS)
    target = new_gold + "<|im_end|>"
    enc = processor(text=[chat(img, prompt(r["question"], new_opts)) + target], images=[img], return_tensors="pt")
    n_tgt = len(tok.encode(target, add_special_tokens=False))
    labels = torch.full_like(enc["input_ids"], -100); labels[0, -n_tgt:] = enc["input_ids"][0, -n_tgt:]
    enc["labels"] = labels
    return enc

CHOICE_IDS = []
for ch in "abcd":
    ss = set()
    for x in [ch, ch.upper(), " "+ch, " "+ch.upper()]:
        t = tok.encode(x, add_special_tokens=False)
        if len(t) == 1: ss.add(t[0])
    CHOICE_IDS.append(sorted(ss))

@torch.inference_mode()
def score_once(img, q, opts):
    inp = processor(text=[chat(img, prompt(q, opts))], images=[img], return_tensors="pt").to("cuda")
    with torch.autocast("cuda", dtype=torch.bfloat16):
        try: lg = model(**inp, logits_to_keep=1, use_cache=False).logits[0, -1].float()
        except TypeError: lg = model(**inp, use_cache=False).logits[0, -1].float()
    return F.softmax(torch.stack([torch.logsumexp(lg[i], 0) for i in CHOICE_IDS]), 0).cpu().numpy()

def score(r):
    img = fit(Image.open(r["path"]).convert("RGB"), INFER_MAX_PIXELS)
    orig = [str(r[c]) for c in "abcd"]; acc = np.zeros(4)
    for k in SHIFTS:
        order = [(j + k) % 4 for j in range(4)]             # 표시 위치 j에 원래 선지 order[j]
        p = score_once(img, r["question"], [orig[i] for i in order])
        for j, i in enumerate(order): acc[i] += p[j]
    return acc / len(SHIFTS)

In [ ]:
# ===== ① 점검: 제로샷 생성 3문항 + 라벨 토큰 =====
model.eval()
for _, r in valid_df.head(3).iterrows():
    img = fit(Image.open(r["path"]).convert("RGB"), INFER_MAX_PIXELS)
    text = chat(img, prompt(r["question"], [str(r[c]) for c in "abcd"]))
    inp = processor(text=[text], images=[img], return_tensors="pt").to("cuda")
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
        out = model.generate(**inp, max_new_tokens=8, do_sample=False, use_cache=True)
    print("정답", r["answer"], "| 생성:", repr(tok.decode(out[0, inp["input_ids"].shape[1]:])), "| 확률", score(r).round(2))
print("프롬프트 끝:", repr(text[-60:]))
e = build_train_example(train_df.iloc[0])
print("라벨 토큰:", repr(tok.decode(e["labels"][0][e["labels"][0] != -100])), "| 시퀀스 길이", e["input_ids"].shape[1])
print("→ 생성이 글자 하나, 프롬프트 끝이 빈 <think></think>, 라벨이 'x<|im_end|>'면 정상")

In [ ]:
# ===== ② 속도 측정 → 학습 데이터 양 N 자동 결정 =====
TIME_BUDGET_MIN = 120      # 원본은 120분 → 3,784개. 같은 결과를 원하면 ② 뒤에 TRAIN_N = 3784 로 고정
model.train(); ts = []
for i in range(6):
    b = {k: v.to("cuda") for k, v in build_train_example(train_df.iloc[order_all[i]]).items()}
    torch.cuda.synchronize(); t0 = time.time()
    with torch.autocast("cuda", dtype=torch.bfloat16):
        loss = model(**b).loss
    loss.backward(); torch.cuda.synchronize(); ts.append(time.time() - t0)
model.zero_grad(set_to_none=True); torch.cuda.empty_cache()
sec = float(np.median(ts[1:]))
TRAIN_N = int(min(len(order_all), TIME_BUDGET_MIN*60 / (sec*1.1)) // 8 * 8)
print(f"샘플당 {sec:.2f}초 → {TIME_BUDGET_MIN}분이면 {TRAIN_N}개 학습 (전체 {len(order_all)}개) | loss 예시 {loss.item():.3f}")
print("GPU 최대 사용:", round(torch.cuda.max_memory_allocated()/1e9, 1), "GB")

## 학습 (끊기면 모델 셀부터 다시 실행 → ②에서 TRAIN_N을 같은 값으로 직접 지정 후 이 셀)

In [ ]:
GRAD_ACCUM, LR, SAVE_EVERY = 8, 1e-4, 50
order = order_all[:TRAIN_N]
n_updates = math.ceil(len(order) / GRAD_ACCUM)
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
sched = get_cosine_schedule_with_warmup(opt, max(1, int(n_updates*0.03)), n_updates)
for _ in range(START_STEP): sched.step()
model.train(); step = START_STEP; run_loss = 0; t0 = time.time(); start_i = START_STEP*GRAD_ACCUM
print(f"총 {n_updates} step ({len(order)}개), {start_i}번째부터")
for i in range(start_i, len(order)):
    random.seed(SEED + i)
    try:
        b = {k: v.to("cuda") for k, v in build_train_example(train_df.iloc[order[i]]).items()}
        with torch.autocast("cuda", dtype=torch.bfloat16):
            loss = model(**b).loss / GRAD_ACCUM
        loss.backward(); run_loss += loss.item()
    except torch.cuda.OutOfMemoryError:
        opt.zero_grad(set_to_none=True); torch.cuda.empty_cache(); print("OOM skip", i); continue
    if (i+1) % GRAD_ACCUM == 0 or i == len(order)-1:
        torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step(); sched.step(); opt.zero_grad(set_to_none=True); step += 1
        if step % 10 == 0:
            el = time.time()-t0; done = i+1-start_i
            print(f"step {step}/{n_updates} loss {run_loss/10:.4f} | {el/done:.2f}s/샘플, 남은 {el/done*(len(order)-i-1)/60:.0f}분"); run_loss = 0
        if step % SAVE_EVERY == 0 or step == n_updates:
            model.save_pretrained(f"{CKPT_DIR}/step_{step}"); print("저장", step)
print("학습 완료")

## valid 672 → 기존 조합과 비교

In [ ]:
model.eval(); P = ["prob_a","prob_b","prob_c","prob_d"]
def run(df, out_csv, every=50):
    done = pd.read_csv(out_csv) if os.path.exists(out_csv) else pd.DataFrame(columns=["id"]+P)
    todo = df[~df.id.astype(str).isin(set(done.id.astype(str)))]; rows = []; t0 = time.time()
    print(f"완료 {len(done)} / 남음 {len(todo)}")
    for n, (_, r) in enumerate(todo.iterrows(), 1):
        rows.append([r.id, *score(r)])
        if n % every == 0 or n == len(todo):
            done = pd.concat([done, pd.DataFrame(rows, columns=done.columns)], ignore_index=True); done.to_csv(out_csv, index=False); rows = []
            el = time.time()-t0; print(f"{n}/{len(todo)} {el/n:.2f}s/문항, 남은 {el/n*(len(todo)-n)/60:.0f}분")
    return done.set_index("id")

vp = run(valid_df, f"{OUT_DIR}/qwen35_valid_probs.csv")
ids = valid_df.id.values; ans = valid_df.answer.map(dict(zip("abcd", range(4)))).values
q = vp.loc[ids, P].values.astype(float)
pr = lambda f: pd.read_csv(f"{EXP_DIR}/{f}").set_index("id").loc[ids, P].values.astype(float)
old = 0.52*pr("EXP-007_train6042_resolution1280_valid_seed43.csv") + 0.48*pr("EXP-034_qwen7b_language_only_train1600_resolution1280_valid_seed43.csv")
b8, g4 = pr("EXP-102_qwen3-vl-8b-instruct_lora_r16_valid_probs.csv"), pr("EXP-120_gemma4-31b_zeroshot_valid_probs.csv")
ok = lambda p: (p.argmax(1) == ans).sum()
print(f"★ Qwen3.5-27B 파인튜닝 단독 valid: {ok(q)}/672")
print(f"  현재 리더보드 최고(기존·8B·Gemma 각 1/3): {ok((old+b8+g4)/3)}/672  →  Qwen3.5 추가(각 1/4): {ok((old+b8+g4+q)/4)}/672")
print(f"  0.96604 조합(기존·8B 반반): {ok(0.5*old+0.5*b8)}/672  →  Qwen3.5 추가(각 1/3): {ok((old+b8+q)/3)}/672")

## test 6,714 (끊기면 이 셀만 다시 실행)

In [ ]:
tp = run(test_df, f"{OUT_DIR}/qwen35_test_probs.csv", every=100)
print(tp.shape)